# Experiment 2 -- aggregate and plot: heatmaps over (Delta_T, Delta_S)

Reads every per-task CSV written by `run_experiment2.py` into
`Results_simulation/experiment2/raw/` (one file per `(regime, seed)`,
each with `3 methods x 5 Delta_T x 5 Delta_S` rows), aggregates the Monte
Carlo mean error at each `(regime, Delta_T, Delta_S, method)` grid point,
and reproduces the paper's supplementary Figure 6 (Appendix A.1,
"Numerical experiment to explore dependence of clustering performance on
target and source signal strengths"): rows = regimes R1-R3, columns =
{target-only raw error, source-only minus target-only, adaptive minus
target-only}.

Run this after the SLURM array in `Slurm_Scripts/experiment2_heatmap/`
has finished (or partially finished).

This experiment compares target-only, source-only, and the adaptive
selector (Algorithm 2); the heatmap grid below is 3x3 (no pooled-method
column).

In [ ]:
import sys, os, glob

# Hardcoded (rather than relative to "..") because the kernel's cwd isn't
# guaranteed to be this notebook's directory -- e.g. VS Code's Jupyter
# extension often starts kernels from the workspace root instead.
# EDIT: set this to the local path of your clone of this repository.
PROJECT_ROOT = "/path/to/Transfer_clustering"
sys.path.insert(0, os.path.join(PROJECT_ROOT, "Numerical_Experiments", "Experiments_Script"))

# The figure below uses matplotlib's text.usetex=True, which shells out to
# `latex`/`dvipng`. If a TeX Live install isn't already on PATH, set
# TEXLIVE_BIN to its bin/ directory (e.g. the output of `dirname $(which latex)`).
TEXLIVE_BIN = None
if TEXLIVE_BIN and os.path.isdir(TEXLIVE_BIN) and TEXLIVE_BIN not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = TEXLIVE_BIN + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from run_experiment2 import REGIME_ORDER, DELTA_T_GRID, DELTA_S_GRID, REGIMES

In [ ]:
RAW_DIR = os.path.join(PROJECT_ROOT, "Results_simulation", "experiment2", "raw")
COMBINED_DIR = os.path.join(PROJECT_ROOT, "Results_simulation", "experiment2", "combined")
os.makedirs(COMBINED_DIR, exist_ok=True)

paths = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
print(f"Found {len(paths)} raw result files")
assert paths, f"No CSVs found in {RAW_DIR} -- has the SLURM array finished any tasks yet?"

df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
df.head()

In [ ]:
# Sanity check: how many distinct seeds actually landed per (regime, method)?
# If well short of N_SEEDS in run_experiment2.py, the array job hasn't
# finished (or some tasks failed -- check Slurm_Scripts/.../Error_Messages).
df.groupby(["regime", "method"])["seed"].nunique().unstack()

In [ ]:
# Err_M(Delta_T, Delta_S): Monte Carlo mean error per grid point.
err = (
    df.groupby(["regime", "Delta_T", "Delta_S", "method"])["error"]
      .agg(mean_error="mean", se_error=lambda s: s.std(ddof=1) / np.sqrt(len(s)), n="count")
      .reset_index()
)
err.to_csv(os.path.join(COMBINED_DIR, "experiment2_err_by_method.csv"), index=False)

# D_M(Delta_T, Delta_S) := Err_M - Err_target for M in {source, adaptive},
# the relative-improvement quantities plotted in the paper's Figure 6.
wide = err.pivot_table(index=["regime", "Delta_T", "Delta_S"], columns="method", values="mean_error").reset_index()
wide["D_source"] = wide["source"] - wide["target"]
wide["D_adaptive"] = wide["adaptive"] - wide["target"]
wide.to_csv(os.path.join(COMBINED_DIR, "experiment2_summary.csv"), index=False)
wide.head(12)

## Figure: 3x3 heatmap grid

Rows = regimes R1-R3. Columns = target-only raw error (sequential blue
colorscale), source-only minus target-only, adaptive minus target-only
(the latter two diverging, blue = improvement / negative, red = worse /
positive, centered at zero), reproducing the paper's Figure 6. Columns 2-3
share one common color scale across all three regimes; the shared range is
the 98th percentile of |D| across every regime so a handful of extreme
cells don't wash out the rest of the scale -- values beyond that are
clipped and shown via the colorbar's triangular extension arrows.

In [ ]:
def to_grid(sub, value_col):
    """(regime-filtered) long dataframe -> len(DELTA_S_GRID) x len(DELTA_T_GRID) array,
    rows = Delta_S (ascending), cols = Delta_T (ascending), matching the
    plan's axis convention (horizontal = Delta_T, vertical = Delta_S)."""
    pivot = sub.pivot(index="Delta_S", columns="Delta_T", values=value_col)
    pivot = pivot.reindex(index=DELTA_S_GRID, columns=DELTA_T_GRID)
    return pivot.values

# Shared color scale for columns 2-3 (convention 5), robust to outliers (convention 6).
all_D = np.concatenate([wide["D_source"].values, wide["D_adaptive"].values])
D_LIM = float(np.nanpercentile(np.abs(all_D), 98))

# Shared sequential scale for column 1 across regimes.
ERR_MAX = float(wide["target"].max())

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
})

fig, axes = plt.subplots(3, 3, figsize=(13, 11))

col_specs = [
    ("target", r"$\widehat{\mathcal{L}}_{T}$", "Blues", 0.0, ERR_MAX, False),
    ("D_source", r"$\widehat{\mathcal{E}}_S$", "RdBu_r", -D_LIM, D_LIM, True),
    ("D_adaptive", r"$\widehat{\mathcal{E}}_{\mathrm{adp}}$", "RdBu_r", -D_LIM, D_LIM, True),
]

for row, regime in enumerate(REGIME_ORDER):
    sub = wide[wide["regime"] == regime]
    cfg = REGIMES[regime]
    for col, (value_col, title, cmap, vmin, vmax, diverging) in enumerate(col_specs):
        ax = axes[row, col]
        grid = to_grid(sub, value_col)
        im = ax.imshow(grid, origin="lower", aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_xticks(range(len(DELTA_T_GRID)))
        ax.set_xticklabels(DELTA_T_GRID, rotation=45)
        ax.set_yticks(range(len(DELTA_S_GRID)))
        ax.set_yticklabels(DELTA_S_GRID)
        if row == 0:
            ax.set_title(title)
        if col == 0:
            ax.set_ylabel(rf"\texttt{{{regime}}} ($d={cfg['d']}$, $n_T={cfg['n_T']}$, $n_S={cfg['n_S']}$)" + "\n" + r"$\Delta_S$")
        if row == 2:
            ax.set_xlabel(r"$\Delta_T$")
        # One shared colorbar per column (attached to the bottom row).
        if row == 2:
            cbar = fig.colorbar(im, ax=axes[:, col].tolist(), orientation="horizontal",
                                 fraction=0.04, pad=0.08, extend=("both" if diverging else "neither"))
            cbar.ax.tick_params(labelsize=10)

fig.suptitle(r"Experiment 2: error over $(\Delta_T, \Delta_S)$, $\mu = 0.8$ fixed", y=0.93)
fig.savefig(os.path.join(COMBINED_DIR, "experiment2_heatmaps.pdf"), bbox_inches="tight")
plt.show()

## Diagnostic: calibrated C0 stability

As in Experiment 1, `run_experiment2.py` records `C0_used` for every
`adaptive` row (the bootstrap-calibrated constant from
`calibrate_C0_bootstrap`). Since Experiment 2's grid covers a much wider
range of `(Delta_T, Delta_S)` than Experiment 1, this is a useful check on
whether `C0_hat` stays stable across signal strengths too, not just
across `(n_T, d)`.

In [ ]:
c0 = df[df["method"] == "adaptive"].copy()
c0["C0_used"] = pd.to_numeric(c0["C0_used"], errors="coerce")
c0.groupby("regime")["C0_used"].agg(["mean", "std", "count"])